# Ноутбук анализа результатов MD

In [ ]:
import math
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import MDAnalysis as mda
import MDAnalysis.analysis.atomicdistances as ad
from MDAnalysis.lib import distances
from MDAnalysis.analysis import rms, align, dihedrals, pca
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA

In [ ]:
system_path = "7NEI_MHET/" #путь до папки с системой
with_mhet = True #надо ли запускать ячейки обработки траекторий системы с MHET

In [ ]:
Path(system_path+"300K/tables").mkdir(parents=True, exist_ok=True) #создание папок для хранения итоговых таблиц
Path(system_path+"343K/tables").mkdir(parents=True, exist_ok=True)

Подгружаю топологию и траекторию (поправленную), для топологии прописываю формат, который соответсвтует GROMACS (иначе выдаст ошибку, так как по умолчанию ожидает AMBER)

In [ ]:
univ_343 = []
univ_300 = []

for i in [1,2,3,4]:
    univ_300.append(mda.Universe(system_path+"topol.top", system_path+f"300K/rep{i}/production_fit.xtc", topology_format="ITP"))
    univ_343.append(mda.Universe(system_path+"topol.top", system_path+f"343K/rep{i}/production_fit.xtc", topology_format="ITP"))

/home/domain/mlmisha/miniconda3/envs/mda_env/lib/python3.10/site-packages/MDAnalysis/topology/ITPParser.py:646: DeprecationWarning: The elements attribute has been populated by guessing elements from atom types. This behaviour has been temporarily added to the ITPParser as we transition to the new guessing API. This behavior will be removed in release 3.0. Please see issue #4698 for more information. 
  warnings.warn(


### RMSD

RMSD я буду вычислять не по всей структуре, а по интересующим меня остаткам. В первую очередь это остатки триады и оксианионного центра, а еще остатки, обеспечивающие связывание субстрата и некторые важные для термостабильности. Вопрос в том, какие брать в анализ атомы. Обычно, как и с RMSF выше, использую CA, но для активного сайта все-же интересно было бы посмотреть и на боковую цепь. Так что, думаю, надо сделать отдельно по CA атомам, потом по всем или некоторым атомам боковой цепи

Определим выборки атомов для всех случаев

In [ ]:
triad_ca = "resid 131 177 209 and name CA" # C-alpha каталитической триады
subsite_1_ca = "resid 63 132 156 179 and name CA"  # C-alpha остатков W156, I179, F63 и M132 subsite 1, в котором связывается субстрат (гидрофоб. карман)
therm_ca = "resid 185 189 130 and name CA" # C-alpha остатков, вместе с W156 имеющих большую роль в термостабильности белка

triad_sc = "resid 131 177 209 and not backbone and not name H*" # Радикал каталитической триады
subsite_1_sc = "resid 63 132 156 179 and not backbone and not name H*"  # Радикалы остатков W156, I179, F63 и M132 subsite 1, в котором связывается субстрат (гидрофоб. карман)
therm_sc = "resid 185 189 130 and not backbone and not name H*" # Радикал остатков, вместе с W156 имеющих большую роль в термостабильности белка

triad_at = "(resid 131 and name OG) or (resid 177 and name OD1 OD2) or (resid 209 and name ND1 NE2)" #  Отдельные атомы боковой цепи каталитической триады
oa_at = "resid 63 132 and name N"  # Азоты оксианионного центра

In [ ]:
def rmsd_pd (univ, select_groups: list, select_groups_names: list) -> pd.DataFrame: #вычисление RMSD по траектории и трансформация данных в pandas DataFrame
    res = rms.RMSD(univ,  # universe to align
             univ,  # reference universe or atomgroup
             select='backbone',  # group to superimpose and calculate RMSD
             groupselections=select_groups,  # groups for RMSD
             ref_frame=0)  # frame index of the reference
    res.run()
    df = pd.DataFrame(res.rmsd,
                  columns=['Frame', 'Time (ps)', 'Backbone']+select_groups_names)
    
    for group in select_groups_names:
        df[group+"_MA"] = df[group].rolling(25).mean()    
    df["Backbone_MA"] = df["Backbone"].rolling(25).mean()
    return df


def rmsd_prep (univ): #вычисление RMSD по всем интересующим группам в рамках одной траектории
    rmsd_ca = rmsd_pd(univ, [triad_ca, subsite_1_ca, therm_ca], ["Triad CA", "Subs. 1 CA", "Therm. CA"])
    rmsd_sc = rmsd_pd(univ, [triad_sc, subsite_1_sc, therm_sc], ["Triad SC", "Subs. 1 SC", "Therm. SC"])
    rmsd_at = rmsd_pd(univ, [triad_at, oa_at], ["Triad At", "OA At"])
    res_pd = pd.merge(pd.merge(rmsd_ca, rmsd_sc.drop(["Backbone_MA", "Time (ps)", "Frame"], axis = 1), on = "Backbone"), rmsd_at.drop(["Backbone_MA", "Time (ps)", "Frame"], axis = 1), on = "Backbone")
    return res_pd  



In [ ]:
i = 1
for uni in univ_300: #вычисление RMSD для систем при 300K
    res = rmsd_prep(uni)
    res.to_csv(system_path+f'300K/tables/rep{i}_rmsd.tsv', sep = "\t", index = False)
    i+=1
i = 1
for uni in univ_343:#вычисление RMSD для систем при 343K
    res = rmsd_prep(uni)
    res.to_csv(system_path+f'343K/tables/rep{i}_rmsd.tsv', sep = "\t", index = False)
    i+=1

### Расстояние между атомами триады

In [ ]:
def distance_calc (univ_list): #вычисление расстояния между атомами триады
    i = 1
    ser_dict = {}
    asp_dict = {}
    for uni in univ_list:
        ser_his = ad.AtomicDistances(uni.select_atoms("resid 131 and name OG"),
                                     uni.select_atoms("resid 209 and name NE2")).run()
        
        his_asp = ad.AtomicDistances(uni.select_atoms("resid 209 and name ND1"),
                                     uni.select_atoms("resid 177 and name OD1")).run()
        ser_dict[f"Rep{i}"]=ser_his.results.flatten().tolist()
        asp_dict[f"Rep{i}"]=his_asp.results.flatten().tolist()
        i+=1

    ser_df = pd.DataFrame(ser_dict)
    asp_df = pd.DataFrame(asp_dict)
    for rep in ser_dict.keys():
        ser_df[rep+"_MA"] = ser_df[rep].rolling(25).mean() 
        asp_df[rep+"_MA"] = asp_df[rep].rolling(25).mean() 
    return ser_df, asp_df

In [ ]:
ser_dist_300, asp_dist_300 = distance_calc(univ_300)
ser_dist_343, asp_dist_343 = distance_calc(univ_343)

In [ ]:
ser_dist_300.to_csv(system_path+"300K/tables/ser_his.tsv", sep = "\t", index = False)
ser_dist_343.to_csv(system_path+"343K/tables/ser_his.tsv", sep = "\t", index = False)

asp_dist_300.to_csv(system_path+"300K/tables/his_asp.tsv", sep = "\t", index = False)
asp_dist_343.to_csv(system_path+"343K/tables/his_asp.tsv", sep = "\t", index = False)

### RMSF

In [ ]:
def rmsf_prep (univ):
    #RMSF рассчитывается на основе референсной усредненной структуры. 
    #Соответственно, сначала нужно ее сделать. За основу выбирается первый фрейм траектории. 
    #Оценивается положение C-alpha атомов. 
    average = align.AverageStructure(univ, univ, select='protein and name CA',
                                    ref_frame=0).run()
    ref = average.results.universe
    #Вычисление RMSF в MDAnalysis требует предаврительного выравнивания всей траектории на референс
    aligner_343 = align.AlignTraj(univ, ref,
                          select='protein and name CA',
                          in_memory=True).run()
    #Теперь можно запустить вычисление RMSF
    c_alphas = univ.select_atoms('protein and name CA')
    rmsf = rms.RMSF(c_alphas).run()
    return rmsf

def rmsf_table (univ_list): #получение pandas DataFrame с RMSF каждой реплики
    i = 1
    res_dict = {}
    for uni in univ_list:
        resi = rmsf_prep(uni)
        print(i, f"Rep{i}")
        res_dict[f"Rep{i}"]=resi.rmsf
        i+=1
    return res_dict


In [ ]:
rmsf_300 = pd.DataFrame(rmsf_table(univ_300))
rmsf_343 = pd.DataFrame(rmsf_table(univ_343))

rmsf_300.to_csv(system_path+"300K/tables/rmsf.tsv", sep = "\t", index = False)
rmsf_343.to_csv(system_path+"343K/tables/rmsf.tsv", sep = "\t", index = False)